# 03 · From picks to a catalogue: association, location and magnitude

A pick says "something arrived at station X at time t". An **event** needs several picks at
different stations that a single source at one place and time can explain. Finding these
groups is **phase association**.

We use **PyOcto** ([Münchmeyer, 2024, *Seismica*](https://doi.org/10.26443/seismica.v3i1.1130)).
It divides space-time into octree cells. A cell is kept only while enough picks remain whose
travel times are consistent with *some* source inside it. Cells are split recursively until
the source is small enough to be located. PyOcto is fast and handles dense pick sets well,
which suits ML output.

Steps:

1. build a travel-time table from the Aluto 1D velocity model,
2. associate the PN-volpick and PN-instance picks from notebook 02,
3. compare the two catalogues in map view and depth,
4. compute **local magnitudes** with the Ethiopian-rift ML scale.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tesfahiwet-yemane/SeismoML/blob/main/notebooks/03_association.ipynb)

In [ ]:
# Running on Google Colab? This cell installs everything and gets the data (2-4 minutes).
# On your own computer it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    repo = "/content/SeismoML"
    if not os.path.isdir(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tesfahiwet-yemane/SeismoML.git", repo], check=True)
    subprocess.run([sys.executable, f"{repo}/scripts/colab_setup.py"] + "--data --outputs".split(), check=True)
    sys.path.insert(0, f"{repo}/src")
    os.chdir(f"{repo}/notebooks")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from obspy import UTCDateTime

import seismoml as sml
from seismoml import association, catalogue, data, magnitude, picking, plotting

plotting.use_style()
cfg = sml.load_config()
CATALOGUES = ["PN-volpick", "PN-instance", "EQT-volpick"]
picks = {k: picking.read_picks(cfg.outputs / "picks" / f"{k}.csv") for k in CATALOGUES}
{k: len(v) for k, v in picks.items()}

## 1 · Velocity model and travel-time table

We use the 1D model of [Wilks et al. (2017)](https://doi.org/10.1016/j.jvolgeores.2017.04.003).
It combines a regional crustal model with geothermal well logs: slow silicic volcanics and
tuffs over a crystalline basement. Depths are relative to sea level, and the stations sit
1.6–2.0 km above it. PyOcto's 1D tables start at depth 0, so we move the model top (2.5 km
a.s.l.) to the table origin. PyOcto computes each travel time with the receiver at that
datum, then corrects for the station's depth below it using a surface velocity. We set
that velocity to the top layer of the model, which contains all the stations.

In [ ]:
vm = association.read_velocity_model(cfg.velocity_model_file)
fig, ax = plt.subplots(figsize=(4, 5))
z = np.r_[np.repeat(vm["depth_km"].to_numpy(), 2)[1:], 45]
ax.plot(np.repeat(vm["vp"].to_numpy(), 2), z, label="Vp")
ax.plot(np.repeat(vm["vs"].to_numpy(), 2), z, label="Vs")
elev = data.station_table(cfg.path("stations"))["elevation_m"] / 1000
ax.axhspan(-elev.max(), -elev.min(), color="0.85", label="station elevations")
ax.set(ylim=(42, -2.6), xlabel="Velocity (km/s)", ylabel="Depth (km b.s.l.)",
       title="Aluto 1D model (Wilks et al., 2017)")
ax.legend();

In [ ]:
tt_table = association.build_travel_time_table(
    cfg.velocity_model_file, cfg.datum_km, cfg.cache,
    xdist_km=60, zdist_km=45, delta_km=0.1)
vp0, vs0 = association.surface_velocities(cfg.velocity_model_file)
print(tt_table.name, f"| top-layer Vp = {vp0} km/s, Vs = {vs0} km/s")

## 2 · Configure and run the associator

The key parameters, chosen for a dense 11-station network about 20 km across:

* `tolerance = 1.0 s`: allowed travel-time misfit while associating. Hydrothermal paths at
  Aluto are strongly heterogeneous, and the 1D model is only approximate.
* `n_picks = 6`, `n_p_picks = 3`, `n_s_picks = 2`, `n_p_and_s_picks = 2`: the minimum
  evidence for an event, counted in picks (P and S). Six picks can come from as few as three
  stations; `n_p_and_s_picks` asks for at least two stations with both phases.
* `cutoff_km = 30`: picks from stations farther than this from a candidate source are not
  used to associate it.

In [ ]:
table = data.station_table(cfg.path("stations"))
stations = data.active_stations(table, cfg.day)
assoc = association.make_associator(
    cfg.lat_range, cfg.lon_range, cfg.depth_range, cfg.datum_km, tt_table, vp0, vs0,
    tolerance=1.0, cutoff_km=30.0,
    n_picks=6, n_p_picks=3, n_s_picks=2, n_p_and_s_picks=2,
)
station_df = association.stations_frame(stations, assoc, cfg.datum_km)
station_df

In [ ]:
events, assigned = {}, {}
for label in CATALOGUES:
    ev, asg = association.associate(picks[label], station_df, assoc, cfg.datum_km)
    events[label], assigned[label] = ev, asg
    print(f"{label:12s}: {len(ev):4d} events, {len(asg):5d} of {len(picks[label]):5d} picks associated "
          f"({len(asg) / max(len(picks[label]), 1):.0%})")

In [ ]:
events["PN-volpick"].head(10)

## 3 · What did we find?

In [ ]:
fig = plotting.map_and_sections(stations, events, cfg.lat_range, cfg.lon_range,
                                depth_range=cfg.depth_range,
                                title=f"Associated events, {cfg.day.date}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
colours = plotting.model_colours(events)
for label, ev in events.items():
    if len(ev):
        axes[0].hist(ev["depth_km"], bins=np.arange(-2.5, 41, 1), histtype="step", lw=1.6,
                     color=colours[label], label=label, orientation="horizontal")
        n_sta = assigned[label].groupby("event_id")["station"].nunique()
        axes[1].hist(n_sta, bins=np.arange(2.5, 12.5, 1), histtype="step", lw=1.6,
                     color=colours[label], label=label)
axes[0].invert_yaxis()
axes[0].set(xlabel="Events", ylabel="Depth (km b.s.l.)", title="Depth distribution")
axes[1].set(xlabel="Stations with picks", ylabel="Events", title="Network coverage per event")
axes[1].legend();

### Look at an event

Always look at the waveforms. Here is the PN-volpick event with the most picks, shown as a
record section with its associated picks.

In [ ]:
ev_v = events["PN-volpick"]
if len(ev_v):
    best = ev_v.assign(n=ev_v["n_P"] + ev_v["n_S"]).sort_values("n", ascending=False).iloc[0]
    t_ev = UTCDateTime(best["time"].timestamp())
    ev_picks = assigned["PN-volpick"][assigned["PN-volpick"]["event_id"] == best["event_id"]]
    st_ev = data.load_waveforms(cfg.sds, cfg.network, cfg.stations, t_ev - 5, t_ev + 20)
    st_ev.detrend("demean").filter("bandpass", freqmin=2, freqmax=25, zerophase=True)
    dist = {s: float(np.hypot(r.x - best.x, r.y - best.y))
            for s, r in zip(stations["station"], station_df.itertuples())}
    fig = plotting.record_section(st_ev, ev_picks, dist, t_ev - 2, t_ev + 12,
                                  title=f"Event {int(best.event_id)}  {best.time:%H:%M:%S.%f}"[:-4]
                                        + f"  depth {best.depth_km:.1f} km b.s.l.")

## 4 · Local magnitude

We use the Main Ethiopian Rift ML scale of
[Keir et al. (2006)](https://doi.org/10.1785/0120060051):

$$M_L = \log_{10} A + 1.196997\,\log_{10}(r/17) + 0.001066\,(r - 17) + 2.0$$

where $A$ is the zero-to-peak Wood-Anderson amplitude (mm) and $r$ the hypocentral
distance (km). As in Keir et al. (2006), we remove the ARGOS response, simulate a
Wood-Anderson seismometer (period 0.8 s, damping 0.8, static magnification 2800), and
measure each horizontal component separately. The network magnitude is the median of all
component values. No station corrections are applied. Keir et al. found station terms with
a spread of about 0.4 units, so expect some scatter.

In [ ]:
inventory = data.load_inventory(cfg.inventory_file)


def load_station(sta):
    return data.load_waveforms(cfg.sds, cfg.network, sta, cfg.t0, cfg.t1)


amplitudes = {}
for label in CATALOGUES:
    events[label], amplitudes[label] = magnitude.catalogue_magnitudes(
        events[label], assigned[label], stations, inventory, load_station)
events["PN-volpick"][["time", "latitude", "longitude", "depth_km", "n_P", "n_S", "magnitude"]].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
for label, ev in events.items():
    if len(ev):
        ax.scatter(ev["time"], ev["magnitude"], s=18, color=colours[label], label=label,
                   edgecolor="k", lw=0.3)
ax.set(ylabel="ML", title=f"Magnitude through {cfg.day.date}")
ax.legend();

## 5 · Save the catalogues

The PyOcto locations are preliminary. They are good enough for statistics and comparisons,
but a proper study would relocate them, for example with NonLinLoc
(`assoc.to_nonlinloc(...)` writes its phase format) followed by double-difference
relocation.

In [ ]:
cat_dir = cfg.outputs / "catalogues"
cat_dir.mkdir(exist_ok=True)
for label in CATALOGUES:
    ev = events[label].copy()
    catalogue.write_table(ev, cat_dir / f"{label}_events.csv")
    picking.save_picks(assigned[label], cat_dir / f"{label}_assigned_picks.csv")
sorted(p.name for p in cat_dir.iterdir())

## Exercises

1. Raise `n_picks` to 8, then lower it to 4. How do the event counts and the depth
   distribution change? Which events disappear first?
2. Replace the 1D model by a homogeneous one (`pyocto.VelocityModel0D(p_velocity=4.5,
   s_velocity=2.6, tolerance=1.0)`). Where do the locations move?
3. Associate the *union* of PN-volpick and EQT-volpick picks. Is the combined catalogue
   larger than either alone?

**Next:** 04 · Comparison with the reference catalogue (`04_catalogue_comparison.ipynb`)